In [5]:
from nba_api.stats.endpoints import TeamGameLog, LeagueGameFinder
import pandas as pd

def get_team_schedule_with_home_away(team_id, season):
    """
    Get a team's schedule with home/away indicators using the nba_api package.
    
    Parameters:
    team_id (int): NBA team ID (e.g., 1610612747 for Lakers)
    season (str): Season in format "2023-24"
    
    Returns:
    pandas.DataFrame: DataFrame with game schedule and home/away flags
    """
    # Use the TeamSchedule endpoint
    schedule = ScheduleLeagueV2(team_id=team_id, season=season)
    schedule_df = schedule.get_data_frames()[0]
    
    # Add a column to indicate if it's a home game
    #schedule_df['IS_HOME'] = schedule_df['MATCH_UP'].str.contains('vs.')
    
    return schedule_df

In [7]:
get_team_schedule_with_home_away(1610612747, "2023-24")

NameError: name 'ScheduleLeagueV2' is not defined

In [15]:
from nba_api.stats.endpoints import TeamGameLog, LeagueGameFinder
import pandas as pd

def get_all_games_for_team(team_id, season):
    """
    Get all games for a team using LeagueGameFinder.
    
    Parameters:
    team_id (int): NBA team ID (e.g., 1610612747 for Lakers)
    season (str): Season in format "2023-24"
    
    Returns:
    pandas.DataFrame: DataFrame with games and home/away flags
    """
    # Use the LeagueGameFinder endpoint
    game_finder = LeagueGameFinder(team_id_nullable=team_id, season_nullable=season)
    games_df = game_finder.get_data_frames()[0]
    
    # Check if the team is the home team based on the MATCHUP field
    # MATCHUP format examples: "LAL vs. BOS" (home) or "LAL @ BOS" (away)
    games_df['IS_HOME'] = games_df['MATCHUP'].str.contains('vs.')
    
    return games_df

In [27]:
team_id = 1610612747  # Lakers
season = "2023-24"

# Get all games using LeagueGameFinder
all_games_df = get_all_games_for_team(team_id, season)
print("\nAll Games (LeagueGameFinder):")
print(all_games_df[['TEAM_NAME', 'TEAM_ID', 'GAME_DATE', 'MATCHUP', 'IS_HOME', 'WL', 'GAME_ID']])


All Games (LeagueGameFinder):
             TEAM_NAME     TEAM_ID   GAME_DATE      MATCHUP  IS_HOME WL  \
0   Los Angeles Lakers  1610612747  2024-04-29    LAL @ DEN    False  L   
1   Los Angeles Lakers  1610612747  2024-04-27  LAL vs. DEN     True  W   
2   Los Angeles Lakers  1610612747  2024-04-25  LAL vs. DEN     True  L   
3   Los Angeles Lakers  1610612747  2024-04-22    LAL @ DEN    False  L   
4   Los Angeles Lakers  1610612747  2024-04-20    LAL @ DEN    False  L   
..                 ...         ...         ...          ...      ... ..   
90  Los Angeles Lakers  1610612747  2023-10-15  LAL vs. MIL     True  L   
91  Los Angeles Lakers  1610612747  2023-10-13  LAL vs. GSW     True  L   
92  Los Angeles Lakers  1610612747  2023-10-11  LAL vs. SAC     True  W   
93  Los Angeles Lakers  1610612747  2023-10-09  LAL vs. BKN     True  W   
94  Los Angeles Lakers  1610612747  2023-10-07    LAL @ GSW    False  L   

       GAME_ID  
0   0042300155  
1   0042300154  
2   0042300153  


In [21]:
game_finder = LeagueGameFinder(team_id_nullable=team_id, season_nullable=season)
games_df = game_finder.get_data_frames()[0]
games_df

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42023,1610612747,LAL,Los Angeles Lakers,0042300155,2024-04-29,LAL @ DEN,L,240,106,...,0.667,11,30,41,30,7,6,8,12,-2.0
1,42023,1610612747,LAL,Los Angeles Lakers,0042300154,2024-04-27,LAL vs. DEN,W,240,119,...,0.833,8,38,46,23,6,2,11,20,11.0
2,42023,1610612747,LAL,Los Angeles Lakers,0042300153,2024-04-25,LAL vs. DEN,L,240,105,...,0.706,8,30,38,23,8,2,7,16,-7.0
3,42023,1610612747,LAL,Los Angeles Lakers,0042300152,2024-04-22,LAL @ DEN,L,240,99,...,0.769,4,34,38,24,6,3,14,20,-2.0
4,42023,1610612747,LAL,Los Angeles Lakers,0042300151,2024-04-20,LAL @ DEN,L,240,103,...,0.895,6,34,40,22,3,7,12,15,-11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,12023,1610612747,LAL,Los Angeles Lakers,0012300041,2023-10-15,LAL vs. MIL,L,241,97,...,0.741,11,36,47,23,12,10,15,21,-11.0
91,12023,1610612747,LAL,Los Angeles Lakers,0012300034,2023-10-13,LAL vs. GSW,L,240,125,...,0.622,9,32,41,30,13,6,17,19,-4.0
92,12023,1610612747,LAL,Los Angeles Lakers,0012300024,2023-10-11,LAL vs. SAC,W,240,109,...,0.773,4,43,47,25,6,4,20,17,8.0
93,12023,1610612747,LAL,Los Angeles Lakers,0012300012,2023-10-09,LAL vs. BKN,W,240,129,...,0.789,8,40,48,25,8,6,15,27,3.0


In [43]:
from nba_api.stats.endpoints import LeagueGameFinder
from nba_api.stats.static import teams
import pandas as pd
import time
import random

def get_all_teams_games(season):
    """
    Get all games for all teams in the league with home/away indicators.
    Includes random delays between requests to avoid rate limiting.
    
    Parameters:
    season (str): Season in format "2023-24"
    
    Returns:
    pandas.DataFrame: DataFrame with all games for all teams
    """
    # Get all teams
    all_teams = teams.get_teams()
    all_games_list = []
    
    # Loop through each team
    for i, team in enumerate(all_teams):
        team_id = team['id']
        team_name = team['full_name']
        
        # Add a delay to avoid rate limiting (random between 1-2 seconds)
        if i > 0:  # Skip delay for the first request
            delay = 1.0 + random.random()
            print(f"Waiting {delay:.2f} seconds before next request...")
            time.sleep(delay)
        
        print(f"Fetching games for {team_name} (ID: {team_id})...")
        
        # Use LeagueGameFinder to get team's games
        game_finder = LeagueGameFinder(
            team_id_nullable=team_id,
            season_nullable=season,
            league_id_nullable='00'  # NBA games only
        )
        
        try:
            team_games_df = game_finder.get_data_frames()[0]
            
            # Check if we got any games
            if not team_games_df.empty:
                # Add home/away indicator
                team_games_df['IS_HOME'] = team_games_df['MATCHUP'].str.contains('vs.')
                all_games_list.append(team_games_df)
            else:
                print(f"No games found for {team_name}")
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"Error for {team_name} - retrying after 60 seconds")
            time.sleep(30)
            continue 
    
    # Combine all teams' games
    if all_games_list:
        all_games_df = pd.concat(all_games_list, ignore_index=True)
        
        # Remove duplicates (since each game appears twice, once for each team)
        # We can use GAME_ID as the unique identifier
        unique_games_df = all_games_df.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])
        
        return unique_games_df
    else:
        return pd.DataFrame()  # Return empty DataFrame if no games found

def get_team_win_loss_home_away(all_games_df):
    """
    Calculate win-loss records for all teams, broken down by home/away.
    
    Parameters:
    all_games_df (DataFrame): DataFrame with all games
    
    Returns:
    DataFrame: Team records with home and away breakdowns
    """
    # Group by team and calculate records
    team_records = []
    
    for team_id, team_games in all_games_df.groupby('TEAM_ID'):
        team_name = team_games['TEAM_NAME'].iloc[0]
        
        # Overall record
        wins = (team_games['WL'] == 'W').sum()
        losses = (team_games['WL'] == 'L').sum()
        
        # Home record
        home_games = team_games[team_games['IS_HOME']]
        home_wins = (home_games['WL'] == 'W').sum()
        home_losses = (home_games['WL'] == 'L').sum()
        
        # Away record
        away_games = team_games[~team_games['IS_HOME']]
        away_wins = (away_games['WL'] == 'W').sum()
        away_losses = (away_games['WL'] == 'L').sum()
        
        team_records.append({
            'TEAM_ID': team_id,
            'TEAM_NAME': team_name,
            'TOTAL_GAMES': len(team_games),
            'WINS': wins,
            'LOSSES': losses,
            'WIN_PCT': wins / (wins + losses) if (wins + losses) > 0 else 0,
            'HOME_GAMES': len(home_games),
            'HOME_WINS': home_wins,
            'HOME_LOSSES': home_losses,
            'HOME_WIN_PCT': home_wins / (home_wins + home_losses) if (home_wins + home_losses) > 0 else 0,
            'AWAY_GAMES': len(away_games),
            'AWAY_WINS': away_wins,
            'AWAY_LOSSES': away_losses,
            'AWAY_WIN_PCT': away_wins / (away_wins + away_losses) if (away_wins + away_losses) > 0 else 0
        })
    
    return pd.DataFrame(team_records).sort_values('WIN_PCT', ascending=False)

# Usage example
season = "2023-24"

# Get all games for all teams
print(f"Retrieving all NBA games for the {season} season with delay between requests...")
all_league_games = get_all_teams_games(season)

# Display all games (with head to limit output)
print(f"\nTotal NBA games retrieved: {len(all_league_games)}")
print("\nSample of games:")
print(all_league_games[['TEAM_NAME', 'GAME_DATE', 'MATCHUP', 'IS_HOME', 'WL', 'GAME_ID']].head(10))

# Get team records with home/away breakdown
team_records_df = get_team_win_loss_home_away(all_league_games)

Retrieving all NBA games for the 2023-24 season with delay between requests...
Fetching games for Atlanta Hawks (ID: 1610612737)...
Waiting 1.13 seconds before next request...
Fetching games for Boston Celtics (ID: 1610612738)...
Waiting 1.30 seconds before next request...
Fetching games for Cleveland Cavaliers (ID: 1610612739)...
Waiting 1.25 seconds before next request...
Fetching games for New Orleans Pelicans (ID: 1610612740)...
Waiting 1.03 seconds before next request...
Fetching games for Chicago Bulls (ID: 1610612741)...
Waiting 1.75 seconds before next request...
Fetching games for Dallas Mavericks (ID: 1610612742)...
Waiting 1.16 seconds before next request...
Fetching games for Denver Nuggets (ID: 1610612743)...
Waiting 1.98 seconds before next request...
Fetching games for Golden State Warriors (ID: 1610612744)...
Waiting 1.75 seconds before next request...
Fetching games for Houston Rockets (ID: 1610612745)...
Waiting 1.66 seconds before next request...
Fetching games for L

In [51]:
def get_multi_season_data(start_season="2020-21", end_season="2023-24"):
    """
    Retrieve NBA game data for multiple seasons.
    
    Parameters:
    start_season (str): First season to retrieve (e.g., "2020-21")
    end_season (str): Last season to retrieve (e.g., "2023-24")
    
    Returns:
    DataFrame: Combined data for all seasons
    """
    # List of seasons to retrieve
    seasons = []
    start_year = int(start_season.split("-")[0])
    end_year = int(end_season.split("-")[0])
    
    for year in range(start_year, end_year + 1):
        season = f"{year}-{str(year+1)[-2:]}"
        seasons.append(season)
    
    print(f"Will retrieve data for {len(seasons)} seasons: {', '.join(seasons)}")
    
    # Store all season data
    all_seasons_data = []
    
    # Process each season
    for season in seasons:
        print(f"\n{'='*50}")
        print(f"PROCESSING SEASON: {season}")
        print(f"{'='*50}\n")
        
        # Get games for this season
        season_data = get_all_teams_games(season)
        
        if not season_data.empty:
            print(f"Successfully retrieved {len(season_data)} games for season {season}")
            all_seasons_data.append(season_data)
            
            # Save individual season data
            season_filename = f"nba_games_{season}.csv"
            season_data.to_csv(season_filename, index=False)
            print(f"Saved data to {season_filename}")
            
            # Calculate and save season records
            season_records = get_team_win_loss_home_away(season_data, season)
            records_filename = f"nba_team_records_{season}.csv"
            season_records.to_csv(records_filename, index=False)
            print(f"Saved team records to {records_filename}")
            
            # Add a longer delay between seasons to avoid rate limiting
            if season != seasons[-1]:  # If not the last season
                delay = 10 + random.random() * 5
                print(f"Waiting {delay:.2f} seconds before processing next season...")
                time.sleep(delay)
        else:
            print(f"No data retrieved for season {season}")
    
    # Combine all seasons
    if all_seasons_data:
        all_games = pd.concat(all_seasons_data, ignore_index=True)
        print(f"\nTotal games across all seasons: {len(all_games)}")
        
        # Save combined data
        all_games.to_csv("nba_all_games_multi_season.csv", index=False)
        
        # Calculate combined records
        all_records = get_team_win_loss_home_away(all_games)
        all_records.to_csv("nba_team_records_all_seasons.csv", index=False)
        
        return all_games
    else:
        print("No data retrieved for any season")
        return pd.DataFrame()

In [53]:
print("Starting NBA data retrieval for multiple seasons")
all_data = get_multi_season_data(start_season="2020-21", end_season="2023-24")

if not all_data.empty:
    print("\nData retrieval complete!")
    print(f"Total games retrieved: {len(all_data)}")
    print(f"Seasons included: {all_data['SEASON'].unique()}")
    print(f"Teams included: {all_data['TEAM_NAME'].nunique()}")
else:
    print("Data retrieval failed.")

Starting NBA data retrieval for multiple seasons
Will retrieve data for 4 seasons: 2020-21, 2021-22, 2022-23, 2023-24

PROCESSING SEASON: 2020-21

Fetching games for Atlanta Hawks (ID: 1610612737)...
Waiting 1.71 seconds before next request...
Fetching games for Boston Celtics (ID: 1610612738)...
Waiting 1.73 seconds before next request...
Fetching games for Cleveland Cavaliers (ID: 1610612739)...
Waiting 1.96 seconds before next request...
Fetching games for New Orleans Pelicans (ID: 1610612740)...
Waiting 2.00 seconds before next request...
Fetching games for Chicago Bulls (ID: 1610612741)...
Waiting 1.59 seconds before next request...
Fetching games for Dallas Mavericks (ID: 1610612742)...
Waiting 1.84 seconds before next request...
Fetching games for Denver Nuggets (ID: 1610612743)...
Waiting 1.76 seconds before next request...
Fetching games for Golden State Warriors (ID: 1610612744)...
Waiting 1.56 seconds before next request...
Fetching games for Houston Rockets (ID: 1610612745)

TypeError: get_team_win_loss_home_away() takes 1 positional argument but 2 were given

In [56]:
from nba_api.stats.endpoints import LeagueGameFinder
from nba_api.stats.static import teams
import pandas as pd
import time
import random
import json
from requests.exceptions import ReadTimeout

def get_all_teams_games(season):
    """
    Get all games for all teams in the league with home/away indicators.
    Includes random delays between requests to avoid rate limiting.
    
    Parameters:
    season (str): Season in format "2023-24"
    
    Returns:
    pandas.DataFrame: DataFrame with all games for all teams
    """
    # Get all teams
    all_teams = teams.get_teams()
    all_games_list = []
    
    # Loop through each team
    for i, team in enumerate(all_teams):
        team_id = team['id']
        team_name = team['full_name']
        
        # Add a delay to avoid rate limiting (random between 1-2 seconds)
        if i > 0:  # Skip delay for the first request
            delay = 1.0 + random.random()
            print(f"Waiting {delay:.2f} seconds before next request...")
            time.sleep(delay)
        
        print(f"Fetching games for {team_name} (ID: {team_id})...")
        
        try:
            # Use LeagueGameFinder to get team's games
            game_finder = LeagueGameFinder(
                team_id_nullable=team_id,
                season_nullable=season,
                league_id_nullable='00'  # NBA games only
            )
            
            team_games_df = game_finder.get_data_frames()[0]
            
            # Check if we got any games
            if not team_games_df.empty:
                # Add home/away indicator
                team_games_df['IS_HOME'] = team_games_df['MATCHUP'].str.contains('vs.')
                # Add season column for multi-season datasets
                team_games_df['SEASON'] = season
                all_games_list.append(team_games_df)
            else:
                print(f"No games found for {team_name}")
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"Error for {team_name} - retrying after 30 seconds: {e}")
            time.sleep(30)
            try:
                # Second attempt
                game_finder = LeagueGameFinder(
                    team_id_nullable=team_id,
                    season_nullable=season,
                    league_id_nullable='00'
                )
                
                team_games_df = game_finder.get_data_frames()[0]
                
                if not team_games_df.empty:
                    team_games_df['IS_HOME'] = team_games_df['MATCHUP'].str.contains('vs.')
                    team_games_df['SEASON'] = season
                    all_games_list.append(team_games_df)
                    print(f"Successfully retrieved data for {team_name} on second attempt")
            except Exception as e2:
                print(f"Second attempt failed for {team_name}: {e2}")
                continue
    
    # Combine all teams' games
    if all_games_list:
        all_games_df = pd.concat(all_games_list, ignore_index=True)
        
        # Remove duplicates (since each game appears twice, once for each team)
        # We can use GAME_ID as the unique identifier
        unique_games_df = all_games_df.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])
        
        return unique_games_df
    else:
        return pd.DataFrame()  # Return empty DataFrame if no games found

def get_team_win_loss_home_away(all_games_df, season=None):
    """
    Calculate win-loss records for all teams, broken down by home/away.
    
    Parameters:
    all_games_df (DataFrame): DataFrame with all games
    season (str, optional): Filter for specific season
    
    Returns:
    DataFrame: Team records with home and away breakdowns
    """
    # Filter by season if specified
    if season:
        filtered_df = all_games_df[all_games_df['SEASON'] == season]
    else:
        filtered_df = all_games_df
    
    # Group by team and calculate records
    team_records = []
    
    for team_id, team_games in filtered_df.groupby('TEAM_ID'):
        team_name = team_games['TEAM_NAME'].iloc[0]
        
        # Overall record
        wins = (team_games['WL'] == 'W').sum()
        losses = (team_games['WL'] == 'L').sum()
        
        # Home record
        home_games = team_games[team_games['IS_HOME']]
        home_wins = (home_games['WL'] == 'W').sum()
        home_losses = (home_games['WL'] == 'L').sum()
        
        # Away record
        away_games = team_games[~team_games['IS_HOME']]
        away_wins = (away_games['WL'] == 'W').sum()
        away_losses = (away_games['WL'] == 'L').sum()
        
        season_val = season if season else "All Seasons"
        
        team_records.append({
            'SEASON': season_val,
            'TEAM_ID': team_id,
            'TEAM_NAME': team_name,
            'TOTAL_GAMES': len(team_games),
            'WINS': wins,
            'LOSSES': losses,
            'WIN_PCT': wins / (wins + losses) if (wins + losses) > 0 else 0,
            'HOME_GAMES': len(home_games),
            'HOME_WINS': home_wins,
            'HOME_LOSSES': home_losses,
            'HOME_WIN_PCT': home_wins / (home_wins + home_losses) if (home_wins + home_losses) > 0 else 0,
            'AWAY_GAMES': len(away_games),
            'AWAY_WINS': away_wins,
            'AWAY_LOSSES': away_losses,
            'AWAY_WIN_PCT': away_wins / (away_wins + away_losses) if (away_wins + away_losses) > 0 else 0
        })
    
    return pd.DataFrame(team_records).sort_values(['SEASON', 'WIN_PCT'], ascending=[True, False])

def get_multi_season_data(start_season="2020-21", end_season="2023-24"):
    """
    Retrieve NBA game data for multiple seasons.
    
    Parameters:
    start_season (str): First season to retrieve (e.g., "2020-21")
    end_season (str): Last season to retrieve (e.g., "2023-24")
    
    Returns:
    DataFrame: Combined data for all seasons
    """
    # List of seasons to retrieve
    seasons = []
    start_year = int(start_season.split("-")[0])
    end_year = int(end_season.split("-")[0])
    
    for year in range(start_year, end_year + 1):
        season = f"{year}-{str(year+1)[-2:]}"
        seasons.append(season)
    
    print(f"Will retrieve data for {len(seasons)} seasons: {', '.join(seasons)}")
    
    # Store all season data
    all_seasons_data = []
    
    # Process each season
    for season in seasons:
        print(f"\n{'='*50}")
        print(f"PROCESSING SEASON: {season}")
        print(f"{'='*50}\n")
        
        # Get games for this season
        season_data = get_all_teams_games(season)
        
        if not season_data.empty:
            print(f"Successfully retrieved {len(season_data)} games for season {season}")
            all_seasons_data.append(season_data)
            
            # Save individual season data
            season_filename = f"nba_games_{season}.csv"
            season_data.to_csv(season_filename, index=False)
            print(f"Saved data to {season_filename}")
            
            # Calculate and save season records
            season_records = get_team_win_loss_home_away(season_data, season)
            records_filename = f"nba_team_records_{season}.csv"
            season_records.to_csv(records_filename, index=False)
            print(f"Saved team records to {records_filename}")
            
            # Add a longer delay between seasons to avoid rate limiting
            if season != seasons[-1]:  # If not the last season
                delay = 10 + random.random() * 5
                print(f"Waiting {delay:.2f} seconds before processing next season...")
                time.sleep(delay)
        else:
            print(f"No data retrieved for season {season}")
    
    # Combine all seasons
    if all_seasons_data:
        all_games = pd.concat(all_seasons_data, ignore_index=True)
        print(f"\nTotal games across all seasons: {len(all_games)}")
        
        # Save combined data
        all_games.to_csv("nba_all_games_multi_season.csv", index=False)
        
        # Calculate combined records
        all_records = get_team_win_loss_home_away(all_games)
        all_records.to_csv("nba_team_records_all_seasons.csv", index=False)
        
        return all_games
    else:
        print("No data retrieved for any season")
        return pd.DataFrame()

# Run the multi-season data retrieval
print("Starting NBA data retrieval for multiple seasons")
all_data = get_multi_season_data(start_season="2020-21", end_season="2023-24")

if not all_data.empty:
    print("\nData retrieval complete!")
    print(f"Total games retrieved: {len(all_data)}")
    print(f"Seasons included: {all_data['SEASON'].unique()}")
    print(f"Teams included: {all_data['TEAM_NAME'].nunique()}")
else:
    print("Data retrieval failed.")

Starting NBA data retrieval for multiple seasons
Will retrieve data for 4 seasons: 2020-21, 2021-22, 2022-23, 2023-24

PROCESSING SEASON: 2020-21

Fetching games for Atlanta Hawks (ID: 1610612737)...
Waiting 1.11 seconds before next request...
Fetching games for Boston Celtics (ID: 1610612738)...
Waiting 1.45 seconds before next request...
Fetching games for Cleveland Cavaliers (ID: 1610612739)...
Waiting 1.84 seconds before next request...
Fetching games for New Orleans Pelicans (ID: 1610612740)...
Waiting 1.75 seconds before next request...
Fetching games for Chicago Bulls (ID: 1610612741)...
Waiting 1.44 seconds before next request...
Fetching games for Dallas Mavericks (ID: 1610612742)...
Waiting 1.98 seconds before next request...
Fetching games for Denver Nuggets (ID: 1610612743)...
Waiting 1.39 seconds before next request...
Fetching games for Golden State Warriors (ID: 1610612744)...
Waiting 1.09 seconds before next request...
Fetching games for Houston Rockets (ID: 1610612745)

In [62]:
from sqlalchemy import create_engine
from requests.exceptions import ReadTimeout
import random
import time 
import pandas as pd
import requests
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

filename = f"team_game_home_away"
all_data.to_sql(filename, engine, if_exists='replace', index=False)
print(f"Finished gathering all data")

Finished gathering all data


In [49]:
all_league_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,IS_HOME
0,52023,1610612737,ATL,Atlanta Hawks,0052300111,2024-04-17,ATL @ CHI,L,241,116,...,8,26,34,30,4,2,9,16,-15.0,False
1,22023,1610612737,ATL,Atlanta Hawks,0022301188,2024-04-14,ATL @ IND,L,241,115,...,9,23,32,25,6,5,15,12,-42.0,False
2,22023,1610612737,ATL,Atlanta Hawks,0022301178,2024-04-12,ATL @ MIN,L,240,106,...,9,31,40,23,4,1,14,25,-3.0,False
3,22023,1610612737,ATL,Atlanta Hawks,0022301159,2024-04-10,ATL vs. CHA,L,240,114,...,7,31,38,35,7,2,16,20,-1.0,True
4,22023,1610612737,ATL,Atlanta Hawks,0022301147,2024-04-09,ATL vs. MIA,L,292,111,...,17,42,59,28,13,2,15,23,-6.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2771,22023,1610612766,CHA,Charlotte Hornets,0022300063,2023-10-25,CHA vs. ATL,W,240,116,...,12,39,51,34,5,3,19,21,6.0,True
2772,12023,1610612766,CHA,Charlotte Hornets,0012300060,2023-10-19,CHA vs. BOS,L,241,99,...,10,39,49,24,8,5,24,17,-28.0,True
2773,12023,1610612766,CHA,Charlotte Hornets,0012300038,2023-10-15,CHA vs. OKC,W,241,117,...,8,35,43,28,10,7,12,15,2.0,True
2774,12023,1610612766,CHA,Charlotte Hornets,0012300025,2023-10-12,CHA @ WAS,L,241,92,...,16,50,66,19,9,9,23,24,-6.0,False


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

filename = f"team_game_home_away"
player_stats_df.to_sql(filename, engine, if_exists='replace', index=False)
print(f"Finished gathering all data for all players for all games in {season}")